# 07 — Оценка агрегаторов на test-группах

Вся логика — в `research/scripts/eval_groups.py`: NDCG, bootstrap-CI, срез по
размеру группы, paired-сравнения audio vs id. Здесь только запуск и графики.

In [ ]:
# Colab: раскомментировать. Локально ячейка не нужна.
# from google.colab import userdata
# token = userdata.get('git')
# !git clone -q https://$token@github.com/Vladislavbro/music-recommendations.git
# %cd music-recommendations
# !pip install -q uv && uv pip install --system -e ".[research]"


In [ ]:
from pathlib import Path

from huggingface_hub import hf_hub_download

HF_REPO = 'Vladislavbro-500/music-recommendations'
ARTIFACTS = Path.cwd() / 'artifacts'

NEEDED = [
    'gsasrec/item_id_to_idx.pkl',
    'user_scores_cache/scores.parquet',
    'audio/embeddings.npy',
    'audio/user_profiles.npy',
    'audio/uid_to_row.pkl',
    'aggregators/groups_split.pkl',
    'aggregators/agree/best.pt',
    'aggregators/groupim/best.pt',
    'aggregators/audio_agree/best.pt',
    'aggregators/group_cross_attn/best.pt',
]
for rel in NEEDED:
    if (ARTIFACTS / rel).exists():
        continue
    hf_hub_download(repo_id=HF_REPO, repo_type='dataset', filename=rel, local_dir=str(ARTIFACTS))
print('artifacts ready')


## Запуск

В `artifacts/eval_results/` появятся `summary.csv`, `summary_by_size.csv`, `paired.csv`, `per_sample.npz`, `summary_table.tex`, `run.json`.

In [ ]:
CONFIG = 'research/configs/aggregators_50m.yaml'

!uv run python research/scripts/eval_groups.py --config {CONFIG}


In [ ]:
import numpy as np
import pandas as pd

from grouprec.eval.bootstrap import load_per_sample

EVAL_DIR = ARTIFACTS / 'eval_results'
FIG_DIR = Path.cwd() / 'thesis' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

summary = pd.read_csv(EVAL_DIR / 'summary.csv')
paired = pd.read_csv(EVAL_DIR / 'paired.csv')
per_sample, sizes, resample_idx = load_per_sample(EVAL_DIR / 'per_sample.npz')
unique_sizes = sorted(np.unique(sizes).tolist())
summary


## Forest plot: NDCG@10 с 95% CI

Цвет — по семейству: аудио синий, ID оранжевый, тривиальные серый.

In [ ]:
import matplotlib.pyplot as plt

FAMILY = {'AudioAGREE': 'tab:blue', 'GroupCrossAttn': 'tab:blue',
          'AGREE': 'tab:orange', 'GroupIM': 'tab:orange',
          'AVG': 'gray', 'LM': 'gray', 'MP': 'gray'}

means = summary['NDCG@10'].values
errs = np.stack([means - summary['NDCG@10_lo95'], summary['NDCG@10_hi95'] - means])
y = np.arange(len(summary))[::-1]

fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(means, y, xerr=errs, fmt='none', ecolor='gray', elinewidth=1.5, capsize=4)
ax.scatter(means, y, s=60, c=[FAMILY[m] for m in summary['method']], zorder=3)
ax.set_yticks(y, summary['method'])
ax.set_xlabel('NDCG@10 (95% bootstrap CI)')
ax.set_title('Test groups: aggregator comparison')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout(); plt.savefig(FIG_DIR / 'eval_forest_plot.png', dpi=150); plt.show()


## NDCG@10 по размеру группы

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
for method in ['AudioAGREE', 'GroupCrossAttn', 'AGREE', 'GroupIM', 'AVG']:
    values = per_sample[method][10]
    means = [values[sizes == s].mean() for s in unique_sizes]
    ax.plot(unique_sizes, means, 'o-', label=method,
            color=FAMILY[method] if method != 'GroupCrossAttn' else 'tab:cyan')
ax.set_xlabel('размер группы'); ax.set_ylabel('NDCG@10'); ax.set_xticks(unique_sizes)
ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(FIG_DIR / 'eval_ndcg_by_size.png', dpi=150); plt.show()


## Значимость пар audio vs ID

`p_one_sided` — доля bootstrap-выборок, где аудио-метод не лучше.

In [ ]:
def stars(p):
    return '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'

pretty = paired.copy()
pretty['delta'] = pretty.apply(
    lambda r: f"{r['delta_mean']:+.4f} [{r['delta_lo95']:+.4f}, {r['delta_hi95']:+.4f}]", axis=1)
pretty['sig'] = pretty['p_one_sided'].apply(stars)
pretty[['audio_method', 'id_method', 'K', 'delta', 'p_one_sided', 'sig']]


## Залить результаты обратно на HF

In [ ]:
# Colab: раскомментировать.
# import os
# from google.colab import userdata
# os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
# !hf upload Vladislavbro-500/music-recommendations \
#     artifacts/eval_results eval_results \
#     --type dataset --commit-message 'test eval'
